# 04 · Validation against later OIG exclusions

The score's weights are learned on exclusions from the development window
(2022-2023) and its headline is measured on the held-out test window (2024
to the LEIE snapshot). This notebook reads the artifacts from
`partd-risk model` and looks at both windows. See ADR 0007.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from _setup import CFG, OCFG, WH, banner
from partd_risk.evaluation.lift import bootstrap_lift, lift_curve
from partd_risk.pipeline import is_synthetic_warehouse, output_dirs, temporal_split

banner()
DIRS = output_dirs(CFG, is_synthetic_warehouse(WH))
summary = json.loads((DIRS.results / "model_summary.json").read_text())
summary["lift"]["design"], summary["lift"]["selected_score"]

## Development window: learned weights and the model choice

In [ ]:
pd.read_csv(DIRS.results / "learned_weights.csv")

In [ ]:
pd.read_csv(DIRS.results / "score_selection.csv")

## Test window: the held-out result

In [ ]:
risk = WH.read("marts", "fct_prescriber_risk")
split = temporal_split(risk, OCFG.split_date)
test = risk[split.test_mask].assign(y=split.test_y[split.test_mask])
print(f"{test['y'].sum():,} test-window exclusions among {len(test):,} prescribers")

In [ ]:
curve = pd.read_csv(DIRS.results / "lift_curve.csv")
curve.pivot(index="k", columns="score", values="lift")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for name, d in curve.groupby("score"):
    ax.plot(d["k"], d["capture_share"], marker="o", label=name)
ax.plot(curve["k"], curve["k"], color="gray", linewidth=1, label="random")
ax.set_xscale("log")
ax.set_xlabel("share flagged")
ax.set_ylabel("share of test-window exclusions captured")
ax.legend()

## Uncertainty at k = 1%

In [ ]:
lo, hi, samples = bootstrap_lift(
    test["outlier_score"].to_numpy(), test["y"].to_numpy(), OCFG.headline_k, reps=OCFG.bootstrap_reps
)
ax = pd.Series(samples).plot.hist(bins=40)
ax.set_title(f"Bootstrap test-window lift at 1%: 95% CI {lo:.1f}x-{hi:.1f}x")

## Single features in each window

Read this as description, not as a menu: choosing features by their
test-window lift would turn the test set into a training set.

In [ ]:
pd.read_csv(DIRS.results / "single_feature_lift.csv")

## What kinds of exclusions are captured, and how soon?

In [ ]:
pd.read_csv(DIRS.results / "captured_exclusion_themes.csv")

In [ ]:
test[test["is_top_1pct"] & test["y"]]["days_to_window_exclusion"].describe()

## Stability across regions (test window, k = 5%)

In [ ]:
rows = []
for region, d in test.groupby("census_region"):
    if d["y"].any():
        rows.append({"census_region": region, **lift_curve(d["outlier_score"], d["y"], [0.05]).iloc[0]})
pd.DataFrame(rows)